In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import re

# 🔽 Ваши импорты
from readerscf import parse_sdr_file
from get_project_root import get_project_root
from substractdf import subtract_column_min
from center_dataframe import center_dataframe
from estimate_M_from_data import estimate_M_from_data
from estimate_crosstalk_matrix import estimate_crosstalk_matrix
from estimate_M_correlation import estimate_M_correlation,estimate_M_correlation_crostalk
from deconvolve_domnisoru import deconvolve_domnisoru
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe
from estimate_M_goodpeaks_crostalk import estimate_M_goodpeaks_crostalk
from estimate_M_clusters_crostalk import estimate_M_clusters_crostalk
from estimate_M_from_clean_peaks import estimate_M_from_clean_peaks
from divide_matrices_np import divide_matrices_np
from normalize_diagonal import normalize_diagonal
from itertools import combinations
from config import IUPAC, ref_str, color_map
from distance_crostalk import distance_crostalk
from multiply_matrix_with_dataframe import multiply_matrix_with_dataframe,matrix_multiply_with_dataframe

def sanitize_sheet_name(filename: str) -> str:
    """Приводит имя файла к формату, допустимому для листа Excel (макс 31 символ, без спецсимволов)."""
    name = Path(filename).stem  # убираем .srd
    name = re.sub(r'[\\\/\?\*\[\]:]', '', name)
    if len(name) > 31:
        name = name[:28] + "..."
    return name

def save_matrices_to_excel(results: list, output_path: Path):
    """Сохраняет 4 матрицы для каждого файла в отдельные листы одного Excel-файла."""
    if not results:
        print("⚠️ Нет данных для сохранения.")
        return

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        used_names = {}
        
        for res in results:
            # Уникальное имя листа
            base_name = sanitize_sheet_name(res['file'])
            if base_name in used_names:
                used_names[base_name] += 1
                sheet_name = f"{base_name}_{used_names[base_name]}"
            else:
                used_names[base_name] = 0
                sheet_name = base_name

            start_row = 0
            matrices = [
                ('matrixdef',    '🔹 M0: Default'),
                ('matrix1',    '🔹 M1: Estimated'),
                ('matrix2',    '🔹 M2: Crosstalk'),
                ('matrix3',    '🔹 M3: Crosstalk correlation'),
                ('matrix4',    '🔹 M4: Crosstalk cluster'),
                ('matrix5',    '🔹 M5: Crosstalk good peaks'),
                 ('matrix6',    '🔹 M6: Crosstalk init default'),
                ('crosstalk_distances_def',    '🔹 M0: dist Default'),
                ('crosstalk_distances_1',    '🔹 M1:dist Estimated'),
                ('crosstalk_distances_2',    '🔹 M2:dist Crosstalk'),
                ('crosstalk_distances_3',    '🔹 M3:  dist Crosstalk correlation'),
                ('crosstalk_distances_4',    '🔹 M4:dist  Crosstalk cluster'),
                ('crosstalk_distances_5',    '🔹 M6:dist Crosstalk good peaks'),
                ('crosstalk_distances_6',    '🔹 M6:dist Crosstalk init matrixdef'),
                ('crosstalk_distances_def',    '🔹 M5:dist  Crosstalk matrixdef'),
              
            ]

            for mat_key, title in matrices:
                df = pd.DataFrame(res[mat_key])
                df.index = [f'Row {i+1}' for i in range(df.shape[0])]
                df.columns = [f'Col {j+1}' for j in range(df.shape[1])]
                
                # Заголовок матрицы
                pd.DataFrame([[title]]).to_excel(writer, sheet_name=sheet_name, 
                                                 startrow=start_row, index=False, header=False)
                # Сама матрица (через 1 строку после заголовка)
                df.to_excel(writer, sheet_name=sheet_name, 
                            startrow=start_row + 2, index=True, header=True)
                
                start_row += df.shape[0] + 4  # данные + заголовок + отступ + заголовок следующей

            tqdm.write(f"   📄 Лист '{sheet_name}' сохранён")

    print(f"💾 Итоговый файл: {output_path}")

import matplotlib.pyplot as plt
from pathlib import Path

def save_pairs_plot(data, matrix_name: str,folder_name:str, file_name: str, project_root: Path):
    # Получаем все пары колонок (без повторов)
    pairs = list(combinations(data.columns, 2))

    # Рассчитываем сетку subplots
    n_pairs = len(pairs)
    n_cols = 3  # Число столбцов в сетке
    n_rows = (n_pairs + n_cols - 1) // n_cols  # Округление вверх
    """Отрисовывает scatter-матрицу и сохраняет с динамическим именем."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 8))
    fig.suptitle(f"Попарные scatter plots после {matrix_name}", fontsize=14)

    for idx, (x_col, y_col) in enumerate(pairs):
        ax = axes.flatten()[idx]
        ax.scatter(data[x_col], data[y_col], alpha=0.7, color=[color_map[x_col]])
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.grid(True, linestyle='--', alpha=0.5)

    for idx in range(n_pairs, n_rows * n_cols):
        fig.delaxes(axes.flatten()[idx])

    plt.tight_layout()

    # 🔹 Динамический путь: имя метода/pairsplotafter{имя_матрицы}_{имя_файла}.jpeg
    save_dir = Path(project_root) / "results"/ folder_name
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"pairsplotafter{matrix_name}_{Path(file_name).stem}.jpeg"

    plt.savefig(save_path, dpi=300)
    plt.close(fig)  # 🔥 КРИТИЧНО: иначе память заполнится и скрипт упадёт на 10-20 файле


def main():
    project_root = Path(get_project_root())
    print(f"📁 Корень проекта: {project_root}")

    srd_dir = project_root / "files" / "сырые SRD" / "сырые SRD"
    if not srd_dir.is_dir():
        print(f"❌ Папка не найдена: {srd_dir}")
        return

    srd_files = sorted(srd_dir.glob("*.srd"))
    if not srd_files:
        print("⚠️ Файлы .srd не найдены.")
        return

    print(f"📂 Найдено файлов: {len(srd_files)}\n")

    results = []  # 📦 Накопитель результатов

    with tqdm(srd_files, desc="🔄 Обработка SRD", unit="файл") as pbar:
        for srd_path in pbar:
            pbar.set_postfix(file=srd_path.name[:18] + "..." if len(srd_path.name) > 18 else srd_path.name)
            
            try:
                sdr_matrix, sdr_channels, sdr_meta =parse_sdr_file(srd_path)
                matrix = sdr_matrix.to_numpy()
                
                if matrix.shape[0] < 10 or matrix.shape[1] < 4:
                    tqdm.write(f"   ⚠️ Матрица слишком мала, пропуск.")
                    continue
                    
                matrixdef = matrix[6:10, 0:4]
                
                data0 = sdr_channels.loc[:, ['dR110', 'dR6G', 'dTAMRA', 'dROX']].copy()
                data0.columns = ['G', 'A', 'T', 'C']
                data0 = subtract_column_min(data0)
                data0 = center_dataframe(data0, method='percentile', percentile=2)

                data1 = multiply_matrix_with_dataframe(matrixdef, data0)
                resdef=distance_crostalk(data1)
                save_pairs_plot(data1, "matrixdef","defaultcallibration", srd_path.name, project_root)
                
                matrix1 = estimate_M_from_data(
                    raw=data0, dye_order=['G', 'A', 'T', 'C'],
                    min_purity=0.5, peak_height=150,
                    peak_distance=15, peak_prominence=80
                )
                data1 = multiply_matrix_with_dataframe(matrix1, data0)
                save_pairs_plot(data1, "matrix1", "estimate_M_from_data",srd_path.name, project_root)
                res1=distance_crostalk(data1)
                matrix2 = estimate_crosstalk_matrix(data0)
                data1 = multiply_matrix_with_dataframe(matrix2, data0)
                res2=distance_crostalk(data1)
                save_pairs_plot(data1, "matrix2","estimate_crosstalk_matrix", srd_path.name, project_root)
                matrix3=estimate_M_correlation_crostalk(data0,init_M=matrixdef)
                data1 = multiply_matrix_with_dataframe(matrix3, data0)
                res3=distance_crostalk(data1)
                save_pairs_plot(data1, "matrix3","estimate_M_correlation_crostalk", srd_path.name, project_root)
                matrix4=estimate_M_clusters_crostalk(data0)
                data1 = multiply_matrix_with_dataframe(matrix4, data0)
                res4=distance_crostalk(data1)
                save_pairs_plot(data1, "matrix4", "estimate_M_clusters_crostalk",srd_path.name, project_root)
                matrix5=estimate_M_goodpeaks_crostalk(data0)
                data1 = multiply_matrix_with_dataframe(matrix5, data0)
                res5=distance_crostalk(data1)
                save_pairs_plot(data1, "matrix5", "estimate_M_goodpeaks_crostalk",srd_path.name, project_root)
                matrix6 = estimate_crosstalk_matrix(data0,init_M=matrixdef)
                data1 = multiply_matrix_with_dataframe(matrix6, data0)
                save_pairs_plot(data1, "matrix6", "estimate_crostalk_init_matrixdef",srd_path.name, project_root)
                res6=distance_crostalk(data1)
                matrix1=normalize_diagonal(matrix1)
                matrix2=normalize_diagonal(matrix2)
                matrix3=normalize_diagonal(matrix3)
                matrix4=normalize_diagonal(matrix4)
                matrix5=normalize_diagonal(matrix5)
                matrix6=normalize_diagonal(matrix6)
                
               

                # 📥 Сохраняем в память
                results.append({
                    'file': srd_path.name,
                    'matrixdef': matrixdef, 
                    'matrix1': matrix1, 'matrix2': matrix2,'matrix3':matrix3,'matrix4':matrix4,'matrix5':matrix5,'matrix6':matrix6,'crosstalk_distances_def': resdef.to_dict('records'),
                    'crosstalk_distances_1': res1.to_dict('records'),'crosstalk_distances_2': res2.to_dict('records'),'crosstalk_distances_3': res3.to_dict('records'),
                    'crosstalk_distances_4': res4.to_dict('records'),'crosstalk_distances_5': res5.to_dict('records'),'crosstalk_distances_6': res6.to_dict('records'),
                    
                })
                
            except Exception as e:
                tqdm.write(f"   ❌ Ошибка в {srd_path.name}: {e}")

    # 💾 Запись в Excel (папка result)
    output_dir = project_root / "results"/"excel"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "srd_all_matrices.xlsx"
    save_matrices_to_excel(results, output_file)


if __name__ == "__main__":
    main()

📁 Корень проекта: C:\Users\Admin\Documents\GitHub\dnasegnercrosstalk
📂 Найдено файлов: 40



🔄 Обработка SRD:   0%|                                           | 0/40 [00:00<?, ?файл/s, file=2_pGEM_G2_PDMA6_36...]

Найдено пиков: 593
  Итерация 1: max Δ = 0.606663
  Итерация 1:  Δassign = 0.000000
  Итерация 1:  Δfrob = 1.220666
  Итерация 1:  cond = 8.189954
  Итерация 1:  mean purity = 0.477981
  Итерация 1:  mean chastity= 0.629483
  Итерация 2: max Δ = 0.052630
  Итерация 2:  Δassign = 0.000000
  Итерация 2:  Δfrob = 0.075753
  Итерация 2:  cond = 7.769262
  Итерация 2:  mean purity = 0.863087
  Итерация 2:  mean chastity= 0.883241
  Итерация 3: max Δ = 0.006631
  Итерация 3:  Δassign = 0.000000
  Итерация 3:  Δfrob = 0.009377
  Итерация 3:  cond = 7.760342
  Итерация 3:  mean purity = 0.871232
  Итерация 3:  mean chastity= 0.887551
  Сходимость на итерации 4
Найдено пиков: 592
  Итерация 1: max Δ = 0.197101
  Итерация 2: max Δ = 0.251379
  Итерация 3: max Δ = 0.007116
  Сходимость на итерации 5
Число обусловленности: 9.42
Найдено пиков: 592
  Итерация 1: max Δ = 0.047171
  Итерация 1:  cond = 7.924063
  Итерация 2: max Δ = 0.002420
  Итерация 2:  cond = 7.837033
  Итерация 3: max Δ = 0.00054

🔄 Обработка SRD:   2%|▉                                  | 1/40 [00:14<09:33, 14.71s/файл, file=3_pGEM_A3_PDMA6_50...]

Найдено пиков: 876
  Итерация 1: max Δ = 0.616748
  Итерация 1:  Δassign = 0.000000
  Итерация 1:  Δfrob = 0.878917
  Итерация 1:  cond = 3.476713
  Итерация 1:  mean purity = 0.465666
  Итерация 1:  mean chastity= 0.619447
  Итерация 2: max Δ = 0.054587
  Итерация 2:  Δassign = 0.000000
  Итерация 2:  Δfrob = 0.091548
  Итерация 2:  cond = 3.483006
  Итерация 2:  mean purity = 0.753408
  Итерация 2:  mean chastity= 0.808443
  Итерация 3: max Δ = 0.017019
  Итерация 3:  Δassign = 0.000000
  Итерация 3:  Δfrob = 0.021311
  Итерация 3:  cond = 3.580161
  Итерация 3:  mean purity = 0.753195
  Итерация 3:  mean chastity= 0.807699
  Итерация 6: max Δ = 0.000464
  Итерация 6:  Δassign = 0.000000
  Итерация 6:  Δfrob = 0.000545
  Итерация 6:  cond = 3.631645
  Итерация 6:  mean purity = 0.760920
  Итерация 6:  mean chastity= 0.812905
  Сходимость на итерации 7
Найдено пиков: 871
  Итерация 1: max Δ = 0.173016
  Итерация 2: max Δ = 0.098075
  Итерация 3: max Δ = 0.014301
  Сходимость на итерац

...

Найдено пиков: 545
  Итерация 1: max Δ = 0.625062
  Итерация 1:  Δassign = 0.000000
  Итерация 1:  Δfrob = 1.082649
  Итерация 1:  cond = 7.675259
  Итерация 1:  mean purity = 0.465789
  Итерация 1:  mean chastity= 0.625447
  Итерация 2: max Δ = 0.060905
  Итерация 2:  Δassign = 0.000000
  Итерация 2:  Δfrob = 0.101483
  Итерация 2:  cond = 7.346814
  Итерация 2:  mean purity = 0.749737
  Итерация 2:  mean chastity= 0.803445
  Итерация 3: max Δ = 0.005589
  Итерация 3:  Δassign = 0.000000
  Итерация 3:  Δfrob = 0.007165
  Итерация 3:  cond = 7.242531
  Итерация 3:  mean purity = 0.758601
  Итерация 3:  mean chastity= 0.810950
  Сходимость на итерации 5
Найдено пиков: 545
  Итерация 1: max Δ = 0.147420
  Итерация 2: max Δ = 0.047521
  Итерация 3: max Δ = 0.011977
  Сходимость на итерации 5
Число обусловленности: 9.65
Найдено пиков: 545
  Итерация 1: max Δ = 0.072024
  Итерация 1:  cond = 8.326562
  Итерация 2: max Δ = 0.013491
  Итерация 2:  cond = 8.238350
  Итерация 3: max Δ = 0.00150

🔄 Обработка SRD:  98%|█████████████████████████████████▏| 39/40 [09:04<00:12, 12.01s/файл, file=pGEM_4_H2_PDMA6_36...]

Найдено пиков: 525
  Итерация 1: max Δ = 0.629589
  Итерация 1:  Δassign = 0.000000
  Итерация 1:  Δfrob = 1.244931
  Итерация 1:  cond = 8.156872
  Итерация 1:  mean purity = 0.464609
  Итерация 1:  mean chastity= 0.623777
  Итерация 2: max Δ = 0.051672
  Итерация 2:  Δassign = 0.000000
  Итерация 2:  Δfrob = 0.095349
  Итерация 2:  cond = 7.421981
  Итерация 2:  mean purity = 0.813771
  Итерация 2:  mean chastity= 0.840913
  Итерация 3: max Δ = 0.014136
  Итерация 3:  Δassign = 0.000000
  Итерация 3:  Δfrob = 0.020652
  Итерация 3:  cond = 7.353855
  Итерация 3:  mean purity = 0.810573
  Итерация 3:  mean chastity= 0.836381
  Сходимость на итерации 5
Найдено пиков: 525
  Итерация 1: max Δ = 0.148458
  Итерация 2: max Δ = 0.063865
  Итерация 3: max Δ = 0.012331
  Сходимость на итерации 5
Число обусловленности: 9.19
Найдено пиков: 525
  Итерация 1: max Δ = 0.076995
  Итерация 1:  cond = 7.467884
  Итерация 2: max Δ = 0.006916
  Итерация 2:  cond = 7.352004
  Итерация 3: max Δ = 0.00000

🔄 Обработка SRD: 100%|██████████████████████████████████| 40/40 [09:17<00:00, 13.94s/файл, file=pGEM_4_H2_PDMA6_36...]


   📄 Лист '2_pGEM_G2_PDMA6_36' сохранён
   📄 Лист '3_pGEM_A3_PDMA6_50' сохранён
   📄 Лист '3_pGEM_B3_PDMA6_50' сохранён
   📄 Лист '3_pGEM_C3_PDMA6_50' сохранён
   📄 Лист '3_pGEM_D3_PDMA6_50' сохранён
   📄 Лист '3_pGEM_E3_PDMA6_50' сохранён
   📄 Лист '3_pGEM_F3_PDMA6_50' сохранён
   📄 Лист '4_pGEM_1_A2_PDMA6_50' сохранён
   📄 Лист '4_pGEM_1_B2_PDMA6_50' сохранён
   📄 Лист '4_pGEM_2_D2_PDMA6_50' сохранён
   📄 Лист '4_pGEM_3_E2_PDMA6_50' сохранён
   📄 Лист '4_pGEM_3_F2_PDMA6_50' сохранён
   📄 Лист '4_pGEM_4_G2_PDMA6_50' сохранён
   📄 Лист '4_pGEM_4_H2_PDMA6_50' сохранён
   📄 Лист '5_pGEM1_GenSeq_D4_PDMA6_36' сохранён
   📄 Лист '5_pGEM1_GenSeq_E4_PDMA6_36' сохранён
   📄 Лист '5_pGEM1_GenSeq_F4_PDMA6_36' сохранён
   📄 Лист '5_pGEM2_GenSeq_G4_PDMA6_36' сохранён
   📄 Лист '5_pGEM2_GenSeq_H4_PDMA6_36' сохранён
   📄 Лист '6_pGEM_1_A3_PDMA6_36' сохранён
   📄 Лист '6_pGEM_1_B3_PDMA6_36' сохранён
   📄 Лист '6_pGEM_2_C3_PDMA6_36' сохранён
   📄 Лист '6_pGEM_2_D3_PDMA6_36' сохранён
   📄 Лист '6_pGEM_